In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [3]:
import os
import numpy as np
import torch
import pickle
import config

from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from sentence_transformers import CrossEncoder

from src.metric import model_evaluation 
from src.datasets import ResumeJDDataset
from src.cross_encoder_training import compute_batch_scores


In [4]:
from torch.amp import autocast, GradScaler

scaler = GradScaler("cuda")
torch.set_float32_matmul_precision("high")

In [5]:
np.random.seed(config.SEED)
torch.manual_seed(config.SEED)
torch.cuda.manual_seed_all(config.SEED)

In [6]:
path=config.CLEANED_DATA_DIR

In [7]:
with open(os.path.join(path,'train_df.pkl'),'rb') as f:
    train_df=pickle.load(f)
    
with open(os.path.join(path,'val_df.pkl'),'rb') as f:
    val_df=pickle.load(f)
        
with open(os.path.join(path,'test_df.pkl'),'rb') as f:
    test_df=pickle.load(f)
    


In [8]:
BATCH_SIZE=config.CHUNK_BATCH_SIZE

In [17]:
cross_encoder= CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2',num_labels=1,device=config.device)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [10]:
g = torch.Generator()
g.manual_seed(config.SEED)

In [11]:
labels=[float(config.label_to_score[label]) for label in train_df['label']]
train_dataset=ResumeJDDataset(train_df['resume_text'].values,train_df['job_description_text'].values,labels)

train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)

In [12]:
val_labels=[float(config.label_to_score[label]) for label in val_df['label']]
val_dataset=ResumeJDDataset(val_df['resume_text'].values,val_df['job_description_text'].values,val_labels)

val_loader=DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False)

In [13]:
min_delta=0.01
count=0
best_score=float('-inf')
epochs=4
patience=2

In [14]:
resume_chunk_map,jd_chunk_map={},{}

In [ ]:
best_model_path=os.path.join(config.CHUNKED_MODEL_DIR,'cross_encoder_chunked')
os.makedirs(config.CHUNKED_MODEL_DIR,exist_ok=True)

In [18]:
optimizer = torch.optim.AdamW(cross_encoder.parameters(), lr=2e-5)
loss_fn = torch.nn.MSELoss()

In [16]:
for epoch in range(epochs):
    
    print(f"Epoch {epoch+1}/{epochs}")
    cross_encoder.train()
    
    total_loss=0
    
    progress_bar = tqdm(train_loader, desc="Training")
    
    for resumes,jds,labels in progress_bar:
        optimizer.zero_grad()
        
        with autocast(device_type="cuda", dtype=torch.float16):
            
            scores=compute_batch_scores(cross_encoder,resumes,jds,resume_chunk_map,jd_chunk_map)
            labels=labels.to(config.device,dtype=scores.dtype)
            
            loss=loss_fn(scores,labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss+=loss.item()
        
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")
        
    average_loss=total_loss/len(progress_bar)
    print("Average Training Loss:",average_loss)
        
        
    #Validation    
    cross_encoder.eval()
    val_loss=0
    scores=[]
    
    with torch.no_grad():
        val_progress_bar = tqdm(val_loader, desc="Validation")
        
        for resumes,jds,labels in val_progress_bar:
            
            with autocast(device_type="cuda", dtype=torch.float16):
                
                val_scores=compute_batch_scores(cross_encoder,resumes,jds,resume_chunk_map,jd_chunk_map)
                labels=labels.to(config.device,dtype=val_scores.dtype)
                loss=loss_fn(val_scores,labels)
                val_scores=val_scores.cpu().numpy()
                
            val_loss += loss.item()
            scores.extend(val_scores)
            val_progress_bar.set_postfix(loss=f"{loss.item():.4f}")
            
        print("Average Validation Loss:",val_loss/len(val_progress_bar))
        
        metrics=model_evaluation(scores,val_df,"job_description_text")
        print("NDCG:", metrics["ndcg_val"])
        print("MAP:", metrics["map_score"])
        
        final_score=0.6*metrics["ndcg_val"]+0.3*metrics["map_score"]+0.1*metrics["mrr_score"]
        
    if final_score>best_score+min_delta:
        best_score=final_score
        cross_encoder.save(best_model_path)
        count=0
    else:
        count+=1

    if count==patience:
        print("Early stopping triggered.")
        break
                
            
    

Epoch 1/4


Training:   0%|          | 0/780 [00:00<?, ?it/s]

Average Training Loss: 0.7971021388920072


Validation:   0%|          | 0/146 [00:00<?, ?it/s]

Average Validation Loss: 0.15051915076854702
NDCG: 0.6734950466189319
MAP: 0.7391976492420055


...

Training:   0%|          | 0/780 [00:00<?, ?it/s]

Average Training Loss: 0.1279844406562356


Validation:   0%|          | 0/146 [00:00<?, ?it/s]

Average Validation Loss: 0.17011593055771027
NDCG: 0.7292320207344068
MAP: 0.7993352353981509


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
cross_encoder= CrossEncoder(best_model_path,num_labels=1,device=config.device)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [20]:
resume_chunk_map,jd_chunk_map={},{}

In [21]:
cross_encoder.eval()
with torch.no_grad():
    with autocast(device_type="cuda", dtype=torch.float16):
        scores=compute_batch_scores(cross_encoder,test_df['resume_text'].values,
                                    test_df['job_description_text'].values,resume_chunk_map,jd_chunk_map)
        scores=scores.cpu().numpy()
    metrics=model_evaluation(scores,test_df,"job_description_text")

In [22]:
print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])

Spearman: 0.3888136232636342
Top-3 Accuracy: 0.9642857142857143
NDCG: 0.6634629338371048
MAP: 0.7521863197758456
MRR: 0.8404566744730679
